# Citrus: A real-world LSA example

Here is a real-world example of Land Suitability Analysis (LSA) using citrus and the North Island of New Zealand as study case. The goal of this notebook is to illustrate how to setup and perform a full LSA using LSAPy.

## Criteria

The first step is to define the criteria that will be used to evaluate citrus land suitability. Here we are going to use those from the [NZLUSDB project](https://baptistehamon.github.io/nzlusdb/) (NZ Land Use Suitability Database), which are based on the criteria rules defined in the [DataSupermarket dataset](https://landuseopportunities.nz/dataset/citrus-crop-suitability-maps-rules-and-yield-information). They are a total of 12 criteria split into two main categories: soil/terrain and climate. The criteria are listed in the table below:

| Criteria | Category | Indicator |
|----------|----------| --------- |
| Soil Drainage | Soil/Terrain | Soil Drainage Class |
| Soil pH | Soil/Terrain | Soil pH |
| Soil Rooting Depth | Soil/Terrain | Potential Rooting Depth (cm) |
| Soil Plant Available Water | Soil/Terrain | Soil Potential Plant Available Water (mm) |
| Soil Salinity | Soil/Terrain | Salinity |
| Soil Slope | Soil/Terrain | Slope (degrees) |
| Soil Stone Content | Soil/Terrain | Topsoil Gravel Content (%) |
| Rainfall Excess | Climate | Annual total precipitation (mm) |
| Flowering Mean Temperature | Climate | Mean daily temperature between Sep 15 and Nov 15 (°C) |
| Budburst Mean Minimum Temperature | Climate | Mean daily minimum temperature between Aug 15 and Oct 15 (°C) |
| Fruit Growth Mean Maximum Temperature | Climate | Mean daily maximum temperature between Jan 1 and Feb 15 (°C) |
| Harvest Heat Stress | Climate | Number of years with at least one hot week (3 days over 35C in a 7-day period) between Dec 1 and Feb 28 (over 10 years) |

```{note}
1. The periods used for climate indicators correspond to South Hemisphere seasons. Conducting the same analysis in the North Hemisphere would require to shift the periods by 6 months.
2. The NZLUSDB computes the Net Irrigation Requirements (NIR) alongside the suitability, and then assumes that the crop water needs are satisfied. As a result, no crop water requirement criterion is used in the NZLUSDB citrus LSA, as is the case in this example.
```

All the data needed for this case study can be found in the [lsapy-data](https://github.com/baptistehamon/lsapy-data) repository. You can use the `lsapy.tutorial` module to access the data. 

In [ ]:
import matplotlib.pyplot as plt

import lsapy
import lsapy.standardize as lstd
from lsapy import LandSuitabilityAnalysis, SuitabilityCriteria

## Criteria Indicators

### Open the data

In [ ]:
data = lsapy.tutorial.open_dataset("citrus-dataset")
data

The climate-related indicators have an additional `time` dimension. In this example, the `time` dimension corresponds to annual values spanning 5 years (2000-2004).

### Visualize the indicators

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 15))
for i, ax in enumerate(axes.flatten()):
    if i < len(data.data_vars):
        ind = list(data.data_vars)[i]
        # If the variable is a climate variable (i.e., 3D with a time dimension),
        # we have to select only the first time step to have 2D data to plot.
        kwargs = {"time": 0} if "time" in data[ind].dims else {}
        data[ind].isel(kwargs).plot(ax=ax)
        ax.set_title(ind)
    else:
        ax.axis("off")
plt.tight_layout()

## Defining the criteria and Land Suitability Analysis

We can now define the criteria using `SuitabilityCriteria` class. Each criteria is defined by its name, category, and a weight giving its relative importance compared to the other criteria. Moreover, the indicator and the standardization function and its parameters used to evaluate the suitability of each criteria is also defined. The function and parameters used in this example are those defined in the [NZLUSDB project](https://baptistehamon.github.io/nzlusdb/).

In [ ]:
sc = {
    "drainage_class": SuitabilityCriteria(
        name="drainage_class",
        long_name="Soil Drainage Class",
        indicator=data["drainage"],
        weight=1,
        category="soilTerrain",
        func=lstd.discrete,
        fparams={"rules": {0: 0, 1: 0.1, 2: 0.6, 3: 0.9, 4: 1, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}},
    ),
    "ph": SuitabilityCriteria(
        name="ph",
        long_name="Soil pH",
        indicator=data["ph"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2024_eq10,
        fparams={"a": 0.0003683, "b": 5.974, "c": 2.767},
    ),
    "potential_rooting_depth": SuitabilityCriteria(
        name="potential_rooting_depth",
        long_name="Potential Rooting Depth",
        indicator=data["potential_rooting_depth"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -10.27, "b": 0.5506},
    ),
    "profile_total_available_water": SuitabilityCriteria(
        name="profile_total_available_water",
        long_name="Soil Potential Plant Available Water (mm)",
        indicator=data["profile_total_available_water"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -0.6951, "b": 150.1},
    ),
    "salinity": SuitabilityCriteria(
        name="salinity",
        long_name="Salinity",
        indicator=data["salinity"],
        weight=1,
        category="soilTerrain",
        func=lstd.boolean,
        fparams={"op": "<", "thresh": 0.1},
    ),
    "slope": SuitabilityCriteria(
        name="slope",
        long_name="Slope",
        indicator=data["slope"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 3.629, "b": 10.70},
    ),
    "topsoil_gravel_content": SuitabilityCriteria(
        name="topsoil_gravel_content",
        long_name="Topsoil Gravel Content",
        indicator=data["topsoil_gravel_content"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 0.9353, "b": 39.34},
    ),
    "rainfall_excess": SuitabilityCriteria(
        name="rainfall_excess",
        long_name="Rainfall excess: annual total precipitation (mm)",
        indicator=data["prcptot"],
        weight=1,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 0.5805, "b": 2080},
    ),
    "tg_mean": SuitabilityCriteria(
        name="tg_mean",
        long_name="Mean daily temperature between Sep 15 and Nov 15 (°C)",
        indicator=data["tg_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -15.14, "b": 10.85},
    ),
    "tn_mean": SuitabilityCriteria(
        name="tn_mean",
        long_name="Mean daily minimum temperature between Aug 15 and Oct 15 (°C)",
        indicator=data["tn_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -7.315, "b": 2.84},
    ),
    "tx_mean": SuitabilityCriteria(
        name="tx_mean",
        long_name="Mean daily maximum temperature between Jan 1 and Feb 15 (°C)",
        indicator=data["tx_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -5.576, "b": 14.55},
    ),
    "year_with_hot_week": SuitabilityCriteria(
        name="year_with_hot_week",
        long_name=(
            "Number of years with at least one hot week (3 days over 35C in a 7-day period)"
            " between Dec 1 and Feb 28 (over 10 years)"
        ),
        indicator=data["year_with_hot_week"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq3,
        fparams={"a": -2.326, "b": 2.143},
    ),
}

For most of the criteria, a continuous standardization function is used, but for the soil drainage and salinity criteria, a discrete and a boolean functions are used, respectively.

```{note}
For soil drainage, only the five first classes are relevant (i.e., _very poor_, _poor_, _imperfect_, _moderately well_ and _well_ drained soils), and a value of 0 is assigned to all the other classes (5 to 12).
```

In [ ]:
lsa = LandSuitabilityAnalysis(
    land_use="citrus", short_name="citrus_suitability", long_name="Citrus Suitability", criteria=sc
)

## Computing the suitability

### Criteria suitability

We can use the ``suitability_type="criteria"`` option to trigger the computation of the suitability for only the criteria.

In [ ]:
res = lsa.run(suitability_type="criteria")
res = res[[v for v in sc.keys()]]  # just to ensure same order for plotting
res

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 15))
for i, ax in enumerate(axes.flatten()):
    if i < len(res.data_vars):
        var = list(res.data_vars)[i]
        # If the variable is a climate variable (i.e., 3D with a time dimension),
        # we have to select only the first time step to have 2D data to plot.
        kwargs = {"time": 0} if "time" in res[var].dims else {}
        res[var].isel(kwargs).plot(ax=ax)
        ax.set_title(var)
    else:
        ax.axis("off")
plt.tight_layout()

### Category and overall suitability

With ``suitability_type="overall"``, the ``run`` function will by default compute the suitability of the categories and then the overall suitability. Here, we use the ``agg_methods="wgmean"`` option to specify that the weighted geometric mean should be used to aggregate the criteria and categories suitability (see the [aggregation methods](../user-guide/aggregation.md) user guide for more details).

In [ ]:
res = lsa.run(suitability_type="overall", agg_methods="wgmean")  # compute category and overall suitability
res = res[[v for v in sc.keys()] + ["soilTerrain", "climate", "suitability"]]  # reorder
res

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
res["soilTerrain"].plot(ax=ax[0], vmin=0, vmax=1)
ax[0].set_title("Soil and Terrain Suitability")
res["climate"].isel(time=0).plot(ax=ax[1], vmin=0, vmax=1)
ax[1].set_title("Climate Suitability")
res["suitability"].isel(time=0).plot(ax=ax[2], vmin=0, vmax=1)
ax[2].set_title("Overall Suitability")
plt.tight_layout()
plt.show()

### Temporal suitability

We can explore how the suitability changes over time. In this example, we compute the mean suitability for over the 5 years and plot the difference for given years.

In [ ]:
mean = res["suitability"].mean(dim="time")
changes = res["suitability"] - mean
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
mean.plot(ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("2000-2005 Mean")
for i, ax, t in zip(range(1, 4), axes[1:], [2000, 2002, 2005], strict=True):
    changes.isel(time=i - 1).plot(ax=ax, vmin=-0.15, vmax=0.15)
    ax.set_title(f"{t} vs 2000-2005")
plt.tight_layout()